# notebook 17 - hyperparameter search for all models

Dr Talebi made a fair point about the last comparison: I tuned the TFT learning rate (0.0003 -> 0.03, which fixed it) but the LSTM and gradient boosting were each trained once with whatever settings I picked first. So the comparison was not like for like - the TFT had an advantage the others never got.

This notebook runs a grid search for the other models so every method gets the same chance.

Protocol, same for every model:
- identical train / validation / test split (chronological 70/15/15), same 5 cities, same 168h window, same 24h horizon
- configurations are ranked on **validation** error only. The test set is not touched during the search
- for the LSTM the training windows are subsampled with a fixed stride so the grid is tractable on my laptop. The same stride is applied to every configuration, so the comparison between configs is fair
- best config per model is then reported on the test set

TODO: if time allows, widen the LSTM grid to include number of layers and dropout

In [1]:
import os, time, itertools, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings('ignore')
torch.manual_seed(42); np.random.seed(42)
ROOT = r'D:\Final year project'
PROC = os.path.join(ROOT, 'processed')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

device: cpu


In [2]:
df = pd.read_csv(os.path.join(PROC, 'master_all_cities.csv'), parse_dates=['time'], low_memory=False)
df = df.sort_values(['city','time']).reset_index(drop=True)
CITIES = ['Jena','London','New_York','Sydney','Tokyo']
SEQ, H = 168, 24
print(df.shape)

(438240, 45)


## part 1 - gradient boosting

This one is quick so I can search it properly. Grid over learning rate, number of iterations and tree size. Ranked on validation MAE.

In [3]:
FEATS = ['temperature','humidity','dew_point','precipitation','wind_speed','wind_gusts','wind_u','wind_v',
         'pressure_msl','surface_pressure','cloud_cover','cloud_cover_low','cloud_cover_mid','cloud_cover_high',
         'shortwave_radiation','direct_radiation','vapour_pressure_deficit','wet_bulb_temp','water_vapour',
         'soil_temperature','temp_lag_1h','temp_lag_3h','temp_lag_6h','temp_lag_24h','temp_roll_6h',
         'temp_roll_24h','temp_diff_1h','hour_sin','hour_cos','month_sin','month_cos','dayofyear_sin',
         'dayofyear_cos','lat','lon']

Xtr,Ytr,Xva,Yva,Xte,Yte = [],[],[],[],[],[]
for city in CITIES:
    d = df[df['city']==city].reset_index(drop=True)
    T = d['temperature'].values.astype(float)
    F = d[FEATS].values.astype(float)
    n = len(T); a, b = int(n*0.70), int(n*0.85)
    targ = sliding_window_view(T, H)
    o = np.arange(SEQ-1, n-H-1)
    tr_o = o[(o+H) < a]
    va_o = o[(o >= a) & ((o+H) < b)]
    te_o = o[o >= b]
    Xtr.append(F[tr_o]); Ytr.append(targ[tr_o+1])
    Xva.append(F[va_o]); Yva.append(targ[va_o+1])
    Xte.append(F[te_o]); Yte.append(targ[te_o+1])

Xtr=np.concatenate(Xtr); Ytr=np.concatenate(Ytr)
Xva=np.concatenate(Xva); Yva=np.concatenate(Yva)
Xte=np.concatenate(Xte); Yte=np.concatenate(Yte)
# subsample train for search speed, same rows for every config
sub = np.arange(0, len(Xtr), 4)
print('train', Xtr[sub].shape, 'val', Xva.shape, 'test', Xte.shape)

train (76453, 35) val (65615, 35) test (65615, 35)


In [4]:
gbm_grid = list(itertools.product([0.05, 0.1, 0.2], [200, 400], [31, 63]))
rows = []
for lr, iters, leaves in gbm_grid:
    t0 = time.time()
    m = MultiOutputRegressor(HistGradientBoostingRegressor(
            learning_rate=lr, max_iter=iters, max_leaf_nodes=leaves,
            early_stopping=True, random_state=42), n_jobs=2)
    m.fit(Xtr[sub], Ytr[sub])
    vm = mean_absolute_error(Yva.flatten(), m.predict(Xva).flatten())
    rows.append({'learning_rate':lr,'max_iter':iters,'max_leaf_nodes':leaves,
                 'val_MAE':round(vm,4),'mins':round((time.time()-t0)/60,1)})
    print(rows[-1], flush=True)

gbm_res = pd.DataFrame(rows).sort_values('val_MAE').reset_index(drop=True)
print()
print(gbm_res.to_string(index=False))
gbm_res.to_csv(os.path.join(PROC,'search_gbm.csv'), index=False)

{'learning_rate': 0.05, 'max_iter': 200, 'max_leaf_nodes': 31, 'val_MAE': 1.3245, 'mins': 1.5}


{'learning_rate': 0.05, 'max_iter': 200, 'max_leaf_nodes': 63, 'val_MAE': 1.2944, 'mins': 1.9}


{'learning_rate': 0.05, 'max_iter': 400, 'max_leaf_nodes': 31, 'val_MAE': 1.2897, 'mins': 2.1}


{'learning_rate': 0.05, 'max_iter': 400, 'max_leaf_nodes': 63, 'val_MAE': 1.2733, 'mins': 3.6}


{'learning_rate': 0.1, 'max_iter': 200, 'max_leaf_nodes': 31, 'val_MAE': 1.2956, 'mins': 1.1}


...

{'learning_rate': 0.2, 'max_iter': 200, 'max_leaf_nodes': 31, 'val_MAE': 1.301, 'mins': 0.8}


{'learning_rate': 0.2, 'max_iter': 200, 'max_leaf_nodes': 63, 'val_MAE': 1.301, 'mins': 1.1}


{'learning_rate': 0.2, 'max_iter': 400, 'max_leaf_nodes': 31, 'val_MAE': 1.2979, 'mins': 1.2}


{'learning_rate': 0.2, 'max_iter': 400, 'max_leaf_nodes': 63, 'val_MAE': 1.3023, 'mins': 1.3}



 learning_rate  max_iter  max_leaf_nodes  val_MAE  mins
          0.05       400              63   1.2733   3.6
          0.10       400              63   1.2742   2.1
          0.10       400              31   1.2774   1.5
          0.10       200              63   1.2821   1.7
          0.05       400              31   1.2897   2.1
          0.05       200              63   1.2944   1.9
          0.10       200              31   1.2956   1.1
          0.20       400              31   1.2979   1.2
          0.20       200              63   1.3010   1.1
          0.20       200              31   1.3010   0.8
          0.20       400              63   1.3023   1.3
          0.05       200              31   1.3245   1.5


In [5]:
# refit the winner and score it on the test set
best = gbm_res.iloc[0]
m = MultiOutputRegressor(HistGradientBoostingRegressor(
        learning_rate=float(best.learning_rate), max_iter=int(best.max_iter),
        max_leaf_nodes=int(best.max_leaf_nodes), early_stopping=True, random_state=42), n_jobs=2)
m.fit(Xtr[sub], Ytr[sub])
gbm_test = mean_absolute_error(Yte.flatten(), m.predict(Xte).flatten())
print('best GBM config:', dict(best))
print('GBM tuned test MAE:', round(gbm_test,4), ' (untuned was 1.258)')

best GBM config: {'learning_rate': np.float64(0.05), 'max_iter': np.float64(400.0), 'max_leaf_nodes': np.float64(63.0), 'val_MAE': np.float64(1.2733), 'mins': np.float64(3.6)}
GBM tuned test MAE: 1.2633  (untuned was 1.258)


## part 2 - LSTM

The important one, since this is what the TFT is being compared against. Grid over learning rate and hidden size. Windows are strided so each config trains in a few minutes - same stride for all of them.

In [6]:
CONT = ['temperature','humidity','dew_point','precipitation','wind_speed','wind_gusts','wind_u','wind_v',
        'pressure_msl','surface_pressure','cloud_cover','cloud_cover_low','cloud_cover_mid','cloud_cover_high',
        'shortwave_radiation','direct_radiation','vapour_pressure_deficit','wet_bulb_temp','water_vapour','soil_temperature']
CYC = ['hour_sin','hour_cos','month_sin','month_cos','dayofyear_sin','dayofyear_cos']
TIDX = 0
onehot = pd.get_dummies(df['city']).astype(float)[CITIES]

def build():
    tr,va,te = [],[],[]
    for city in CITIES:
        ix = np.where((df['city']==city).values)[0]
        n=len(ix); a,b = int(n*0.70), int(n*0.85)
        tr.append(ix[:a]); va.append(ix[a:b]); te.append(ix[b:])
    sc = StandardScaler().fit(df.iloc[np.concatenate(tr)][CONT].values)
    def mk(rows):
        return np.concatenate([sc.transform(df.iloc[rows][CONT].values),
                               df.iloc[rows][CYC].values,
                               onehot.iloc[rows].values], axis=1).astype(np.float32)
    return (np.concatenate([mk(r) for r in tr]),
            np.concatenate([mk(r) for r in va]),
            [(c, mk(r)) for c,r in zip(CITIES,te)], sc)

TRAIN, VAL, TEST, SCALER = build()
N_IN = len(CONT)+len(CYC)+len(CITIES)
print('input dim', N_IN, '| train rows', TRAIN.shape[0])

input dim 31 | train rows 306765


In [7]:
class WD(Dataset):
    def __init__(s,d): s.d=torch.FloatTensor(d)
    def __len__(s): return len(s.d)-SEQ-H+1
    def __getitem__(s,i): return s.d[i:i+SEQ], s.d[i+SEQ:i+SEQ+H, TIDX]

class LSTMNet(nn.Module):
    def __init__(s, n_in, hidden, layers=2, dropout=0.2):
        super().__init__()
        s.l = nn.LSTM(n_in, hidden, layers, batch_first=True, dropout=dropout)
        s.n = nn.LayerNorm(hidden)
        s.h = nn.Sequential(nn.Linear(hidden,64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64,H))
    def forward(s,x):
        o,_ = s.l(x); return s.h(s.n(o[:,-1]))

STRIDE = 20     # same for every config
MAXEP  = 6

def run_config(lr, hidden, stride=STRIDE, epochs=MAXEP, patience=2, verbose=False):
    tr_ds = WD(TRAIN); va_ds = WD(VAL)
    tr = DataLoader(Subset(tr_ds, range(0,len(tr_ds),stride)), batch_size=256, shuffle=True, drop_last=True)
    va = DataLoader(Subset(va_ds, range(0,len(va_ds),stride)), batch_size=256)
    torch.manual_seed(42)
    net = LSTMNet(N_IN, hidden).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
    lf = nn.HuberLoss()
    best, bad, state = 1e9, 0, None
    for ep in range(epochs):
        net.train()
        for xb,yb in tr:
            xb,yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); loss = lf(net(xb), yb); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step()
        net.eval(); v=0
        with torch.no_grad():
            for xb,yb in va: v += lf(net(xb.to(DEVICE)), yb.to(DEVICE)).item()
        v /= max(1,len(va))
        if verbose: print(f'   ep{ep} val {v:.4f}', flush=True)
        if v < best: best, bad, state = v, 0, {k:t.clone() for k,t in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(state)
    return best, net

print('ready')

ready


In [8]:
lstm_grid = list(itertools.product([1e-4, 1e-3, 3e-3, 1e-2], [64, 128]))
rows = []
for lr, hid in lstm_grid:
    t0 = time.time()
    v, _ = run_config(lr, hid)
    rows.append({'learning_rate':lr,'hidden_size':hid,'val_loss':round(v,4),'mins':round((time.time()-t0)/60,1)})
    print(rows[-1], flush=True)

lstm_res = pd.DataFrame(rows).sort_values('val_loss').reset_index(drop=True)
print()
print(lstm_res.to_string(index=False))
lstm_res.to_csv(os.path.join(PROC,'search_lstm.csv'), index=False)

{'learning_rate': 0.0001, 'hidden_size': 64, 'val_loss': 0.0546, 'mins': 2.1}


{'learning_rate': 0.0001, 'hidden_size': 128, 'val_loss': 0.0387, 'mins': 4.9}


{'learning_rate': 0.001, 'hidden_size': 64, 'val_loss': 0.0286, 'mins': 1.7}


{'learning_rate': 0.001, 'hidden_size': 128, 'val_loss': 0.0274, 'mins': 4.2}


{'learning_rate': 0.003, 'hidden_size': 64, 'val_loss': 0.0279, 'mins': 1.6}


{'learning_rate': 0.003, 'hidden_size': 128, 'val_loss': 0.0265, 'mins': 4.0}


{'learning_rate': 0.01, 'hidden_size': 64, 'val_loss': 0.0284, 'mins': 1.7}


{'learning_rate': 0.01, 'hidden_size': 128, 'val_loss': 0.0333, 'mins': 2.0}



 learning_rate  hidden_size  val_loss  mins
        0.0030          128    0.0265   4.0
        0.0010          128    0.0274   4.2
        0.0030           64    0.0279   1.6
        0.0100           64    0.0284   1.7
        0.0010           64    0.0286   1.7
        0.0100          128    0.0333   2.0
        0.0001          128    0.0387   4.9
        0.0001           64    0.0546   2.1


In [9]:
# retrain the winning LSTM config with more data (smaller stride) and score on test
b = lstm_res.iloc[0]
print('best LSTM config:', dict(b))
_, net = run_config(float(b.learning_rate), int(b.hidden_size), stride=6, epochs=12, patience=3, verbose=True)

tm, ts = SCALER.mean_[TIDX], SCALER.scale_[TIDX]
net.eval(); per_city = {}
for city, arr in TEST:
    dl = DataLoader(WD(arr), batch_size=256)
    ps, as_ = [], []
    with torch.no_grad():
        for xb, yb in dl:
            ps.append(net(xb.to(DEVICE)).cpu().numpy()); as_.append(yb.numpy())
    p = np.concatenate(ps)*ts+tm; a = np.concatenate(as_)*ts+tm
    per_city[city] = mean_absolute_error(a.flatten(), p.flatten())
lstm_test = float(np.mean(list(per_city.values())))
print()
print('per city:', {k:round(v,3) for k,v in per_city.items()})
print('LSTM tuned test MAE:', round(lstm_test,4), ' (untuned was 1.39)')

best LSTM config: {'learning_rate': np.float64(0.003), 'hidden_size': np.float64(128.0), 'val_loss': np.float64(0.0265), 'mins': np.float64(4.0)}


   ep0 val 0.0297


   ep1 val 0.0257


   ep2 val 0.0251


   ep3 val 0.0242


...

   ep8 val 0.0239


   ep9 val 0.0245


   ep10 val 0.0232


   ep11 val 0.0226



per city: {'Jena': 1.303, 'London': 1.21, 'New_York': 1.527, 'Sydney': 1.18, 'Tokyo': 1.109}
LSTM tuned test MAE: 1.2659  (untuned was 1.39)


In [10]:
summary = pd.DataFrame([
    {'model':'LSTM','tuned_params':f'lr={b.learning_rate}, hidden={int(b.hidden_size)}',
     'configs_tried':len(lstm_grid),'test_MAE':round(lstm_test,4)},
    {'model':'Gradient boosting','tuned_params':f'lr={best.learning_rate}, iters={int(best.max_iter)}, leaves={int(best.max_leaf_nodes)}',
     'configs_tried':len(gbm_grid),'test_MAE':round(gbm_test,4)},
    {'model':'TFT','tuned_params':'lr=0.03, hidden=32','configs_tried':'lr search (3e-4 -> 0.03)','test_MAE':1.2472},
])
print(summary.to_string(index=False))
summary.to_csv(os.path.join(PROC,'search_summary.csv'), index=False)

            model                  tuned_params            configs_tried  test_MAE
             LSTM          lr=0.003, hidden=128                        8    1.2659
Gradient boosting lr=0.05, iters=400, leaves=63                       12    1.2633
              TFT            lr=0.03, hidden=32 lr search (3e-4 -> 0.03)    1.2472


## takeaway

TODO fill in once the numbers are in. The point is that every model now has a documented search behind it, so the final comparison is like for like rather than one tuned model against several untuned ones.